# Music Studio — full browser interface on Kaggle

First create your saved input with **Music-Studio-Dataset.ipynb**. Attach that
ONE **Music Studio Dataset**, select **GPU T4 ×2**, enable **Internet**, and run
the **single startup cell** below. Open **OPEN MUSIC STUDIO** when it appears.
Keep the cell running. Press **Create** in the studio when you are ready;
startup never generates a diagnostic song.

This is the separate YuE2 Studio Linux browser interface: create songs, inspect
and edit scores/MIDI, re-render, cover/restyle recordings, manage a library,
split stems, make MIDI, time karaoke, process audio, create lyric videos,
use LoRAs and drive the studio through MCP. It does not replace the existing
native FP16/ComfyUI notebooks. Generation uses the saved Q8 GGUF model on GPU 0;
the second T4's memory is not pooled. Stems and speech recognition default to CPU.
The local writer, MIDI and style-description helpers also use CPU builds.

This pinned Linux fork has a different feature set from newer Windows releases.
T4 LoRA training, Windows VST plugins, AB2/chord-variety and newer extension/
continuation controls are not provisioned. The guide covers every available
panel plus alternatives and limitations. No subscription/API key is required
for the included local tools; optional external providers have their own terms.

Saved songs, SQLite library, settings, LoRAs and logs live under
`/kaggle/working/music_studio`. Models stay in your read-only Dataset and
temporary runtime state. **Save Version with output files** before ending
your session; a saved draft does not preserve your library. Reattach saved
outputs and follow the guide's restore procedure in a later session.
Keep the Cloudflare link/key private. Stop the previous music/movie/image
startup cell before starting another studio in the same Kaggle session.

[Complete tool guide PDF](https://github.com/bothashaun-maker/Music/blob/main/Music-Studio-Guide.pdf)
· [Setup and troubleshooting](https://github.com/bothashaun-maker/Music/blob/main/docs/music-studio/README.md)


In [ ]:
DATASET_PATH = 'auto'  # Or the exact saved Music Studio Dataset directory under /kaggle/input.
GPU_INDEX = 0
RESTORE_LIBRARY_PATH = 'auto'  # One attached saved music_studio library; use 'none' for a fresh library.

# Self-contained support; no repository clone or separate repair cell required.
import base64, hashlib, json, os, signal, subprocess, sys, zlib
from pathlib import Path
packed = base64.b64decode('...')
assert hashlib.sha256(packed).hexdigest() == '7da3cfdde193a473cf8410ae6b75d07f1f9dff547d1d18826412f73cb946cfd9', 'Embedded Music Studio source checksum failed'
support = Path('/kaggle/working/.music_studio_support/7da3cfdde193')
for relative, contents in json.loads(zlib.decompress(packed)).items():
    path = support / relative
    if not path.resolve().is_relative_to(support.resolve()):
        raise RuntimeError('Invalid embedded source path')
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(contents)
task_env = dict(os.environ)
task_env['PYTHONPATH'] = str(support) + os.pathsep + task_env.get('PYTHONPATH', '')
task_env['MUSIC_STUDIO_DATASET_PATH'] = DATASET_PATH
task_env['MUSIC_STUDIO_GPU'] = str(GPU_INDEX)
task_env['MUSIC_STUDIO_RESTORE_LIBRARY'] = RESTORE_LIBRARY_PATH

task = subprocess.Popen([sys.executable, '-u', '-m', 'music_studio.launch'], env=task_env,
                        stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                        text=True, bufsize=1, start_new_session=True)
try:
    for line in task.stdout:
        print(line, end='', flush=True)
    code = task.wait()
    if code:
        raise RuntimeError('Music Studio stopped with exit code ' + str(code) + '; inspect the error above and saved logs.')
except KeyboardInterrupt:
    print('Stopping this Music Studio process. Saved songs are retained.', flush=True)
finally:
    if task.poll() is None:
        os.killpg(task.pid, signal.SIGTERM)
        try:
            task.wait(timeout=25)
        except subprocess.TimeoutExpired:
            os.killpg(task.pid, signal.SIGKILL)
            task.wait(timeout=5)
